In [1]:
import simulator
from NoisyCircuits.utils.CreateNoiseModel import CreateNoiseModel
from NoisyCircuits.utils.BuildQubitGateModel import BuildModel
from NoisyCircuits.QuantumCircuit import QuantumCircuit
import numpy as np
import time
import pennylane as qml

2026-05-29 16:45:23,901	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
import pickle

with open("test_model.pkl", "rb") as f:
    data = pickle.load(f)

single = data["s"]
double = data["d"]
meas = data["m"]

In [2]:
noise_model = CreateNoiseModel("../noise_models/Sample_Noise_Model_Heron_QPU.csv", [["sx", "x", "rx", "rz"], ["cz", "rzz"]]).create_noise_model()

In [3]:
modeller = BuildModel(noise_model, 32, 10, 1e-6, [["sx", "x", "rx", "rz"], ["cz", "rzz"]], False)

In [4]:
res = modeller.build_qubit_gate_model()

In [6]:
from pympler.asizeof import asizeof

print("Size Post-processed noise: {:.4f} MB".format(asizeof(res) / 1024 / 1024))

Size Post-processed noise: 3.6892 MB


In [5]:
import measurement_error_applicator

In [6]:
meas = res[2]

In [9]:
q = 16
probs_array = np.random.rand(2**q)
probs_array = probs_array / np.sum(probs_array)
probs_array_np = np.copy(probs_array)
t0 = time.perf_counter_ns()
measurement_error_applicator.apply_measurement_error(probs_array, meas, [i for i in range(q)], q, 2)
t1 = time.perf_counter_ns()
op = meas[q-1]
for i in range(q-2, -1, -1):
    op = np.kron(op, meas[i])
t2 = time.perf_counter_ns()
probs_array_np = np.dot(op, probs_array_np)
t3 = time.perf_counter_ns()
print("Time for C++: {:.4f} ms".format((t1 - t0) / 1e6))
print("Time for NumPy: {:.4f} ms".format((t3 - t2) / 1e6))
print("Building noise matrix: {:.4f} ms".format((t2 - t1) / 1e6))
print("Probability Match: ", np.allclose(probs_array, probs_array_np))
print("Fidelity between results: {:.6f}".format(np.sum(np.sqrt(probs_array * probs_array_np))))

Time for C++: 1.5016 ms
Time for NumPy: 1538.0179 ms
Building noise matrix: 29238.2956 ms
Probability Match:  True
Fidelity between results: 1.000000


In [11]:
q = 32
probs_array = np.random.rand(2**q)
probs_array = probs_array / np.sum(probs_array)
t0 = time.perf_counter_ns()
measurement_error_applicator.apply_measurement_error(probs_array, meas, [i for i in range(q)], q, 2)
t1 = time.perf_counter_ns()
print("Time for C++: {:.4f} ms".format((t1 - t0) / 1e6))

Time for C++: 50302.3528 ms


In [14]:
import pympler.asizeof as asiz

print("Array Size: {:.4f} GB".format(asiz.asizeof(probs_array) / 1024 / 1024 / 1024))
print("Measurement Error Matrix Size: {:.4f} GB".format(asiz.asizeof(op) / 1024 / 1024 / 1024))

Array Size: 32.0000 GB
Measurement Error Matrix Size: 64.0000 GB


In [6]:
qc = QuantumCircuit(3, noise_model, "heron", 100, 10, "qulacs", False, False, 1e-5)

Successfully switched backend to qulacs.


2026-05-28 15:42:18,894	INFO worker.py:2007 -- Started a local Ray instance.
/Users/adam-ukj7r05xnu2fywx/miniconda3/envs/NoisyCircuits/lib/python3.10/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [7]:
qc.H(0)
qc.H(1)
qc.H(2)
for _ in range(4):
    for i in range(3):
        qc.RY(np.random.rand() * np.pi, i)
    for i in range(2):
        qc.CX(i, i+1)

In [8]:
inst_list = qc.instruction_list

In [ ]:
qc.run_pure_state([0, 1, 2])

In [3]:
def build_random_unitary(n):
    N = 1 << n
    random_matrix = np.random.rand(N, N) + 1j * np.random.rand(N, N)
    Q, R = np.linalg.qr(random_matrix)
    D = np.diag(R)
    D = D / np.abs(D)
    return Q @ np.diag(D)

In [4]:
import random

def generate_random_qubit_list(n, b, a=0):
    return random.sample(range(a, b), n)

In [5]:
def test_unitary(U, apply_qubits, dev):
    @qml.qnode(dev)
    def apply_unitary():
        qml.QubitUnitary(U, wires=apply_qubits)
        return qml.state()
    return apply_unitary()

In [11]:
q = 11
dev = qml.device("lightning.qubit", wires=q)
unitary = build_random_unitary(q)
apply_qubits = generate_random_qubit_list(q, q)
instruction_lst = [["unitary", apply_qubits, unitary]]
t0 = time.perf_counter_ns()
state_pennylane = test_unitary(unitary, apply_qubits, dev)
state_custom = np.zeros(1 << q, dtype=np.complex128)
state_custom[0] = 1.0
t1 = time.perf_counter_ns()
simulator.simulate_circuit(instruction_lst, state_custom, single, double, q, False, 1, True, 4)
t2 = time.perf_counter_ns()
state_custom = state_custom.reshape([2]*q).transpose(list(range(q))[::-1]).reshape(-1)
print("States Match: ", np.allclose(state_custom, state_pennylane, atol=1e-10))
print(f"Time taken by Pennylane: {(t1 - t0) / 1e6:.2f} ms")
print(f"Time taken by Custom Simulator: {(t2 - t1) / 1e6:.2f} ms")

States Match:  True
Time taken by Pennylane: 74.72 ms
Time taken by Custom Simulator: 169.38 ms


In [24]:
q = 3
res_vector = np.zeros(1 << q, dtype=np.complex128)
simulator.simulate_circuit(inst_list, res_vector, res[0], res[1], q, False, 1, False, 12)
print("Final state vector:", res_vector)

Final state vector: [0.01182102+0.j 0.60230801+0.j 0.04765409+0.j 0.00064321+0.j
 0.08245996+0.j 0.01197928+0.j 0.12792452+0.j 0.11520991+0.j]


In [30]:
single_cpp = {
    q: {gate: payload["qubit_channel"] for gate, payload in gates.items()}
    for q, gates in res[0].items()
}
two_cpp = {
    gate: {pair: payload["qubit_channel"] for pair, payload in pairs.items()}
    for gate, pairs in res[1].items()
}

In [35]:
res_vector_noise = np.zeros(1 << q, dtype=np.complex128)
simulator.simulate_circuit(inst_list, res_vector_noise, single_cpp, two_cpp, q, True, 1000, False, 10)
print("Final state vector with noise:", res_vector_noise)

Final state vector with noise: [0.01606517+0.j 0.58610093+0.j 0.04893395+0.j 0.00699917+0.j
 0.08268067+0.j 0.01469303+0.j 0.1282698 +0.j 0.11625729+0.j]


In [ ]:
gate_map = {
    "sx" : lambda p, q: qml.SX(q[0]),
    "x" : lambda p, q: qml.X(q[0]),
    "rx" : lambda p, q: qml.RX(p, q[0]),
    "rz" : lambda p, q: qml.RZ(p, q[0]),
    "cz" : lambda p, q: qml.CZ(q),
    "rzz" : lambda p, q: qml.IsingZZ(p, q)
}
dev = qml.device("lightning.qubit", wires=3)
@qml.qnode(dev)
def apply_circuit(inst:list, gate_map:dict=gate_map):
    for ent in inst:
        gate_name, qubits, params = ent
        gate_map[gate_name](params, qubits)
    return qml.state()

state_pennylane = apply_circuit(inst_list)
state_pennylane = state_pennylane.reshape([2]*q).transpose(list(range(q))[::-1]).reshape(-1)

In [22]:
print("States Match: ", np.allclose(res_vector, state_pennylane, atol=1e-10))

States Match:  True


In [23]:
np.linalg.norm(state_pennylane - res_vector)

np.float64(7.904849603204866e-16)